# Lesson 04: OpenAI Embeddings & Semantic Similarity Search

This notebook covers:
1. Initializing OpenAI embeddings using the `text-embedding-3-small` model.
2. Implementing the Cosine Similarity metric from scratch using NumPy.
3. Performing exhaustive pairwise sentence similarity comparisons.
4. Building an end-to-end Semantic Search function to retrieve the most relevant documents.

> For detailed theoretical explanations, mathematical derivations, and architecture diagrams, see the [Lesson Documentation](README.md).

## 1. Environment & Setup
Load environment variables and initialize the embedding model.

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

# Load environment variables from .env
load_dotenv()

openai_key = os.getenv('OPENAI_API_KEY')
print(f"OPENAI_API_KEY configured: {'Yes' if openai_key and len(openai_key) > 10 else 'No (Please set in .env)'}")

# Initialize the fast and cost-effective text-embedding-3-small model
try:
    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    test_vec = embeddings.embed_query('Test query')
    print(f"OpenAI Embeddings initialized successfully! Vector dimension: {len(test_vec)}")
except Exception as exc:
    print(f"Note: OpenAI API request skipped ({exc}). Falling back to remote HuggingFace embeddings for demonstration.")
    from langchain_huggingface import HuggingFaceEndpointEmbeddings
    embeddings = HuggingFaceEndpointEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')


## 2. Cosine Similarity Function
Compute the cosine of the angle between two multi-dimensional vectors:
$$\text{similarity} = \frac{A \cdot B}{\|A\| \|B\|}$$

In [ ]:
def cosine_similarity(a, b) -> float:
    """Calculate cosine similarity score between two vectors."""
    vec_a = np.array(a)
    vec_b = np.array(b)
    
    dot_product = np.dot(vec_a, vec_b)
    norm_a = np.linalg.norm(vec_a)
    norm_b = np.linalg.norm(vec_b)
    
    if norm_a == 0 or norm_b == 0:
        return 0.0
    
    return float(dot_product / (norm_a * norm_b))

# Verify function with orthogonal and identical vectors
print(f"Identical vectors similarity: {cosine_similarity([1, 0], [1, 0]):.1f}")
print(f"Orthogonal vectors similarity: {cosine_similarity([1, 0], [0, 1]):.1f}")


## 3. Pairwise Sentence Comparison
Compare all unique pairs of sentences to examine semantic proximity across domains.

In [ ]:
sentences = [
    'The cat sat on the mat.',
    'A feline rested on the rug.',
    'The dog played in the yard.',
    'I love programming in Python.',
    'Python is my favorite programming language.'
]

# Generate dense embeddings for all sentences in batch
sentence_embeddings = embeddings.embed_documents(sentences)
print(f"Generated {len(sentence_embeddings)} embeddings (each with dimension {len(sentence_embeddings[0])}).\n")

# Compute pairwise similarity across all unique pairs (i, j)
print("--- Pairwise Similarity Comparisons ---")
for i in range(len(sentences)):
    for j in range(i + 1, len(sentences)):
        score = cosine_similarity(sentence_embeddings[i], sentence_embeddings[j])
        print(f"'{sentences[i]}' VS '{sentences[j]}'")
        print(f"  --> Similarity Score: {score:.4f}\n")


## 4. Semantic Search Implementation
Define a custom semantic search function that embeds a query, compares it against a collection of documents, and returns the top-$k$ ranked results.

In [ ]:
def semantic_search(query: str, documents: list[str], embedding_model, top_k: int = 3) -> list[tuple[float, str]]:
    """Retrieve top-k most semantically similar documents for a given query."""
    # 1. Embed query and documents
    query_embedding = embedding_model.embed_query(query)
    doc_embeddings = embedding_model.embed_documents(documents)
    
    # 2. Compute similarity for each document
    similarities = []
    for doc, doc_vec in zip(documents, doc_embeddings):
        score = cosine_similarity(query_embedding, doc_vec)
        similarities.append((score, doc))
    
    # 3. Sort by similarity descending
    similarities.sort(key=lambda item: item[0], reverse=True)
    
    # 4. Return top-k results
    return similarities[:top_k]

# Knowledge base corpus
documents = [
    'LangChain is a framework for developing applications powered by language models.',
    'Python is a high-level programming language.',
    'Machine learning is a subset of artificial intelligence.',
    'Embeddings convert text into numerical vectors.',
    'The weather today is sunny and warm.'
]

print(f"Knowledge base indexed with {len(documents)} documents.")


## 5. Testing Semantic Search
Query the semantic search engine with natural language questions.

In [ ]:
queries = [
    'What is LangChain?',
    'What is embeddings?'
]

for q in queries:
    print(f"\n==============================")
    print(f"Query: '{q}'")
    print(f"==============================")
    top_results = semantic_search(q, documents, embeddings, top_k=3)
    for rank, (score, doc) in enumerate(top_results, 1):
        print(f"Rank #{rank} (Score: {score:.4f}):")
        print(f"  {doc}")
